In [1]:
# !pip install aiomoex
# !pip install tinkoff-investments
# !pip install yfinance

Открытые вопросы:
* Стоит ли описывать варианты получения данных и причину выбора?
* Идея - разбивать на 2 интервал. До СВО и после, что-то вроде микро исследования. Периоды для обучения - 2-3 года. Прогноз (в идеале) как в исследовании - 30 минутные интервалы с дообучением (ноль понимания как его делать в реалтайм)


Что нужно сделать ближайшее:
* Выбрать инструменты для результирующего сравнения своей стратегии с HODL
* Выбрать инструменты для прогнозирования: металлы, фонды, акции, сырье. Не более 10, желательно даже 5
* Выгрузить IMOEX индекс на периоде для сравнения (в идеале потом можно еще с открытием вкладов сравнить, но сложно кодом считать)
* Реализовать функции расчета индикаторов, которые необходимы будут для обучения


In [5]:
from tinkoff.invest import Client, AsyncClient, CandleInterval, SecurityTradingStatus
from tinkoff.invest.services import InstrumentsService
from tinkoff.invest.utils import quotation_to_decimal, now
from tinkoff.invest.caching.instruments_cache.instruments_cache import InstrumentsCache

import pandas as pd
from dotenv import load_dotenv

import os
# import asyncio
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Optional
import warnings

warnings.filterwarnings("ignore")

In [3]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
load_dotenv()

# TOKEN = "REDACTED"
TOKEN = os.getenv("TOKEN")

'REDACTED'

Подробная информация по тикеру.

**Не работает с индексами вроде индекса МосБиржи**

In [5]:
def get_info(ticker, TOKEN = TOKEN):
    with Client(TOKEN) as client:
        instruments: InstrumentsService = client.instruments
        tickers = []
        for method in ["shares", "bonds", "etfs", "currencies", "futures"]:
            for item in getattr(instruments, method)().instruments:
                tickers.append(
                    {
                        "name": item.name,
                        "ticker": item.ticker,
                        "class_code": item.class_code,
                        "figi": item.figi,
                        "uid": item.uid,
                        "type": method,
                        "min_price_increment": quotation_to_decimal(
                            item.min_price_increment
                        ),
                        "scale": 9 - len(str(item.min_price_increment.nano)) + 1,
                        "lot": item.lot,
                        "trading_status": str(
                            SecurityTradingStatus(item.trading_status).name
                        ),
                        "api_trade_available_flag": item.api_trade_available_flag,
                        "currency": item.currency,
                        "exchange": item.exchange,
                        "buy_available_flag": item.buy_available_flag,
                        "sell_available_flag": item.sell_available_flag,
                        "short_enabled_flag": item.short_enabled_flag,
                        "klong": quotation_to_decimal(item.klong),
                        "kshort": quotation_to_decimal(item.kshort),
                    }
                )

        tickers_df = pd.DataFrame(tickers)

        ticker_df = tickers_df[tickers_df["ticker"] == ticker]
        if ticker_df.empty:
            print("There is no such ticker: %s", ticker)
            # return

        figi = ticker_df["figi"].iloc[0]
        print(f"\nTicker {ticker} have figi={figi}\n")
        print(f"Additional info for this {ticker} ticker:")
        print(ticker_df.iloc[0])

        return figi

ticker = "SPY"
get_info(ticker)


Ticker SPY have figi=BBG000BDTBL9

Additional info for this SPY ticker:
name                                          SPDR S&P 500 ETF Trust
ticker                                                           SPY
class_code                                                     SPBXM
figi                                                    BBG000BDTBL9
uid                             59ba0c48-4f13-429a-b1a1-6aff71baca05
type                                                            etfs
min_price_increment                                       0.01000000
scale                                                              2
lot                                                                1
trading_status              SECURITY_TRADING_STATUS_BREAK_IN_TRADING
api_trade_available_flag                                        True
currency                                                         usd
exchange                                                     unknown
buy_available_flag            

'BBG000BDTBL9'

Получить только FIGI.

In [29]:
def get_figi(ticker: str) -> str:
    ticker = ticker.upper()
    with Client(TOKEN) as client:
        # find_instrument ищет по всей базе Тинькофф
        response = client.instruments.find_instrument(query=ticker)

        print(response)
        
        for instrument in response.instruments:
            # СТРОГАЯ ПРОВЕРКА: тикер в ответе должен быть равен запрошенному
            if instrument.ticker == ticker:
                # Дополнительно выведем тип, чтобы вы видели, что скачиваете
                print(f"Подтверждено: {ticker} это {instrument.instrument_type} (FIGI: {instrument.figi})")
                return instrument.figi
                
        raise ValueError(f"Инструмент с тикером {ticker} не найден в базе.")

ticker = "SBER"
get_figi(ticker)

FindInstrumentResponse(instruments=[InstrumentShort(isin='RU000A1025U5', figi='BBG00XR1B5V7', ticker='RU000A1025U5', class_code='TQCB', instrument_type='bond', name='Сбер Банк 001Р-SBER17', uid='f6f3414f-848a-4132-9ae6-2775dff48429', position_uid='b2d0709e-bde2-41de-b3f1-72fe477668e7', instrument_kind=<InstrumentType.INSTRUMENT_TYPE_BOND: 1>, api_trade_available_flag=False, for_iis_flag=True, first_1min_candle_date=datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), first_1day_candle_date=datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), for_qual_investor_flag=False, weekend_flag=False, blocked_tca_flag=False, lot=1), InstrumentShort(isin='RU000A109X37', figi='TCS50A109X37', ticker='RU000A109X37', class_code='A36', instrument_type='bond', name='Сбербанк ПАО 001Р-SBERD3', uid='7b326a24-9f3d-4247-bc16-41479368775c', position_uid='59b33ecb-0cb4-4d66-9ebe-721637ea4593', instrument_kind=<InstrumentType.INSTRUMENT_TYPE_BOND: 1>, api_trade_available_flag=False,

'TCS704730N88'

In [25]:
tickers_to_test = ["SBER", "GLDRUB_TOM", "CNYRUB_TOM", "BRH6"]

for t in tickers_to_test:
    try:
        figi = get_figi(t)
        print(f"Успех! {t} -> {figi}")
    except Exception as e:
        print(f"Ошибка для {t}: {e}")

Подтверждено: SBER это share (FIGI: TCS704730N88)
Успех! SBER -> TCS704730N88
Подтверждено: GLDRUB_TOM это currency (FIGI: BBG000VJ5YR4)
Успех! GLDRUB_TOM -> BBG000VJ5YR4
Подтверждено: CNYRUB_TOM это currency (FIGI: BBG0013HRTL0)
Успех! CNYRUB_TOM -> BBG0013HRTL0
Подтверждено: BRH6 это futures (FIGI: FUTBR0326000)
Успех! BRH6 -> FUTBR0326000


In [ ]:
def get_5min(ticker: str) -> pd.DataFrame:
    ticker = ticker.upper()
    file_path = DATA_DIR / f"{ticker}_5min.parquet"

    # Если файл есть — читаем и сразу делаем DateTime колонкой
    if file_path.exists():
        print(f"Загружаю из кэша: {ticker}")
        df = pd.read_parquet(file_path)
        if df.index.name == "DateTime":
            df = df.reset_index()
        return df.sort_values("DateTime", ascending=False)

    # Если файла нет — скачиваем
    print(f"Скачиваю {ticker} с 01.01.2025...")
    figi = get_figi(ticker)
    new_rows = []
    current = datetime(2025, 1, 1)
    now = datetime.now()

    with Client(TOKEN) as client:
        while current < now:
            end = min(current + timedelta(days=7), now - timedelta(minutes=10))

            if end <= current:
                break

            for _ in range(3):
                try:
                    candles = client.market_data.get_candles(
                        figi=figi,
                        from_=current,
                        to=end,
                        interval=CandleInterval.CANDLE_INTERVAL_5_MIN
                    ).candles

                    for c in candles:
                        new_rows.append({
                            "DateTime": c.time,
                            "Open":  c.open.units  + c.open.nano  / 1e9,
                            "High":  c.high.units  + c.high.nano  / 1e9,
                            "Low":   c.low.units   + c.low.nano   / 1e9,
                            "Close": c.close.units + c.close.nano / 1e9,
                            "Volume": c.volume
                        })
                    print(f"  +{len(candles)} свечей → {end.strftime('%Y-%m-%d %H:%M')}")
                    break
                except Exception as e:
                    if "RESOURCE_EXHAUSTED" in str(e):
                        print("Лимит! Жду 45 сек...")
                        time.sleep(45)
                    else:
                        time.sleep(5)
            else:
                print(f"Не удалось скачать кусок {current.date()} → {end.date()}")
            
            current = end
            time.sleep(1.2)

    if not new_rows:
        raise RuntimeError(f"Не удалось скачать данные для {ticker}")

    df = pd.DataFrame(new_rows)
    df = df.sort_values("DateTime", ascending=False)
    
    df.to_parquet(file_path, compression="zstd", index=False)
    print(f"Сохранено: {file_path} → {len(df):,} строк")

    return df

In [13]:
# Словарь настроек интервалов: (Константа API, Максимальный шаг скачивания)
INTERVALS = {
    "5min":  (CandleInterval.CANDLE_INTERVAL_5_MIN, timedelta(days=7)),
    "15min": (CandleInterval.CANDLE_INTERVAL_15_MIN, timedelta(days=21)),
    "1hour": (CandleInterval.CANDLE_INTERVAL_HOUR, timedelta(days=31)),
    "4hour": (CandleInterval.CANDLE_INTERVAL_4_HOUR, timedelta(days=31 * 2)),
    "1day":  (CandleInterval.CANDLE_INTERVAL_DAY, timedelta(days=365))
}

def get_candles_data(ticker: str, 
                     interval_name: str = "5min",
                     start_date: datetime = datetime(2025, 1, 1), 
                     end_date: Optional[datetime] = None) -> pd.DataFrame:
    """
    Загружает исторические свечи, используя локальное кэширование в Parquet.
    Решает проблему смешивания tz-aware и tz-naive дат.
    """
    ticker = ticker.upper()
    if interval_name not in INTERVALS:
        raise ValueError(f"Неверный интервал. Доступные: {list(INTERVALS.keys())}")
    
    t_interval, chunk_step = INTERVALS[interval_name]
    file_path = DATA_DIR / f"{ticker}_{interval_name}.parquet"
    
    # 1. Нормализация дат (приводим всё к naive - без часовых поясов для сравнения)
    if start_date.tzinfo is not None:
        start_date = start_date.replace(tzinfo=None)
    
    if end_date is None:
        end_date = datetime.now()
    elif end_date.tzinfo is not None:
        end_date = end_date.replace(tzinfo=None)

    df = pd.DataFrame()

    # 2. Проверка локального кэша
    if file_path.exists():
        df = pd.read_parquet(file_path)
        # Очищаем кэшированные данные от TZ для корректного сравнения
        df['DateTime'] = pd.to_datetime(df['DateTime']).dt.tz_localize(None)
        
        if not df.empty:
            cache_min = df['DateTime'].min()
            cache_max = df['DateTime'].max()
            
            # Если кэш покрывает запрошенный период (с запасом в 1 час)
            if cache_min <= start_date and cache_max >= (end_date - timedelta(minutes=60)):
                print(f"[{ticker}] Данные полностью взяты из кэша.")
                mask = (df['DateTime'] >= start_date) & (df['DateTime'] <= end_date)
                return df.loc[mask].sort_values("DateTime", ascending=False)
            
            print(f"[{ticker}] Кэш неполный. Докачиваю недостающее...")

    # 3. Скачивание данных через API
    figi = get_figi(ticker) # Убедитесь, что функция get_figi определена
    new_rows = []
    
    # Для API Тинькофф нужны даты с часовым поясом (UTC)
    current_download = start_date.replace(tzinfo=timezone.utc)
    target_end = end_date.replace(tzinfo=timezone.utc)
    
    with Client(TOKEN) as client:
        while current_download < target_end:
            chunk_end = min(current_download + chunk_step, target_end)

            for attempt in range(3):
                try:
                    candles = client.market_data.get_candles(
                        figi=figi,
                        from_=current_download,
                        to=chunk_end,
                        interval=t_interval
                    ).candles

                    for c in candles:
                        new_rows.append({
                            "DateTime": c.time,
                            "Open":  c.open.units  + c.open.nano  / 1e9,
                            "High":  c.high.units  + c.high.nano  / 1e9,
                            "Low":   c.low.units   + c.low.nano   / 1e9,
                            "Close": c.close.units + c.close.nano / 1e9,
                            "Volume": c.volume
                        })
                    break 
                except Exception as e:
                    if "RESOURCE_EXHAUSTED" in str(e):
                        print("Лимит запросов! Жду 45 сек...")
                        time.sleep(45)
                    else:
                        print(f"Ошибка: {e}. Пробую снова...")
                        time.sleep(5)
            
            current_download = chunk_end
            time.sleep(0.5) # Небольшая пауза между чанками

    # 4. Объединение, очистка и сохранение
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        # Сразу очищаем новые данные от TZ
        new_df['DateTime'] = pd.to_datetime(new_df['DateTime']).dt.tz_localize(None)
        
        df = pd.concat([df, new_df]).drop_duplicates(subset=['DateTime'])
    
    if df.empty:
        raise RuntimeError(f"Не удалось получить данные для {ticker}")

    # Сортируем и сохраняем полный кэш
    df = df.sort_values("DateTime", ascending=False)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(file_path, compression="zstd", index=False)
    
    # Возвращаем только запрошенный диапазон
    mask = (df['DateTime'] >= start_date) & (df['DateTime'] <= end_date)
    return df.loc[mask]

In [14]:
# Получить дневки для Газпрома
df_daily = get_candles_data("GAZP", interval_name="1day", start_date=datetime(2023, 1, 1))

# Получить часовики для Сбера
df_hourly = get_candles_data("SBER", interval_name="1hour")

[GAZP] Кэш неполный. Докачиваю недостающее...


In [ ]:
def clean_market_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищает данные от пропусков и нулевых значений.
    """
    df = df.copy()
    
    # 1. Заменяем чистые нули на NaN (в цене Close/Open нулей быть не может)
    # Делаем это только для колонок с ценами
    cols_to_fix = ['Open', 'High', 'Low', 'Close']
    for col in cols_to_fix:
        df[col] = df[col].replace(0, pd.NA)
    
    # 2. Сортируем по времени (важно для правильного заполнения "вперед")
    df = df.sort_values("DateTime")
    
    # 3. Forward Fill: заполняем пропуски последним известным значением
    df[cols_to_fix] = df[cols_to_fix].ffill()
    
    # 4. Backward Fill: на случай, если пропуски в самых первых строках
    df[cols_to_fix] = df[cols_to_fix].bfill()
    
    # 5. Обработка объема (Volume)
    # Для объема пропуски лучше заменять на 0, так как сделок просто не было
    df['Volume'] = df['Volume'].fillna(0)
    
    return df

,DateTime,Open,High,Low,Close,Volume
1820,2026-01-17 00:00:00,27.00,27.00,27.00,27.00,0
1819,2026-01-16 20:00:00,26.81,26.88,26.77,26.85,317124
1818,2026-01-16 19:00:00,26.85,26.94,26.77,26.82,285400
1817,2026-01-16 18:00:00,27.09,27.09,26.84,26.85,57021
1816,2026-01-16 17:00:00,27.07,27.14,27.07,27.09,57462
...,...,...,...,...,...,...
4,2025-01-02 18:00:00,27.12,27.40,27.03,27.39,153397
3,2025-01-02 17:00:00,27.36,27.36,27.10,27.14,277640
2,2025-01-02 16:00:00,27.53,27.56,27.37,27.37,149390
1,2025-01-02 15:00:00,27.38,27.54,27.27,27.52,445410


In [ ]:
def fill_time_gaps(df: pd.DataFrame, interval_name: str) -> pd.DataFrame:
    """
    Вставляет пропущенные временные интервалы (строки), которых нет в данных.
    """
    # Маппинг для метода resample
    resample_map = {
        "5min": "5min", "15min": "15min", "1hour": "h", "1day": "D"
    }
    freq = resample_map.get(interval_name, "5min")
    
    df = df.set_index("DateTime").sort_index()
    
    # Создаем полный индекс без дыр от начала до конца
    full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
    
    # Переиндексируем (появятся строки с NaN там, где были дыры)
    df = df.reindex(full_range)
    df.index.name = "DateTime"
    
    # Заполняем пустоты
    df[['Open', 'High', 'Low', 'Close']] = df[['Open', 'High', 'Low', 'Close']].ffill()
    df['Volume'] = df['Volume'].fillna(0)
    
    return df.reset_index()

In [ ]:
def add_technical_indicators(df: pd.DataFrame, 
                             sma_periods: list = [20, 50, 200],
                             ema_periods: list = [12, 26],
                             rsi_period: int = 14,
                             macd_fast: int = 12,
                             macd_slow: int = 26,
                             macd_signal: int = 9) -> pd.DataFrame:
    """
    Добавляет технические индикаторы к датафрейму с OHLCV данными.
    
    Параметры:
        df: DataFrame с колонками ['DateTime', 'Open', 'High', 'Low', 'Close', 'Volume']
        sma_periods: список периодов для SMA
        ema_periods: список периодов для EMA
        rsi_period: период для RSI
        macd_fast, macd_slow, macd_signal: параметры MACD
    
    Возвращает:
        Тот же DataFrame с добавленными колонками индикаторов
    """
    df = df.copy()
    close = df['Close']
    
    # SMA (Simple Moving Average)
    for period in sma_periods:
        df[f'SMA_{period}'] = close.rolling(window=period).mean()
    
    # EMA (Exponential Moving Average)
    for period in ema_periods:
        df[f'EMA_{period}'] = close.ewm(span=period, adjust=False).mean()
    
    # RSI (Relative Strength Index)
    delta = close.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(window=rsi_period, min_periods=rsi_period).mean()
    avg_loss = loss.rolling(window=rsi_period, min_periods=rsi_period).mean()
    
    # Сглаживание по методу Wilder's (аналог ema с alpha = 1/period)
    avg_gain = avg_gain.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()
    avg_loss = avg_loss.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()
    
    rs = avg_gain / avg_loss
    df[f'RSI_{rsi_period}'] = 100 - (100 / (1 + rs))
    
    # MACD
    ema_fast = close.ewm(span=macd_fast, adjust=False).mean()
    ema_slow = close.ewm(span=macd_slow, adjust=False).mean()
    df['MACD'] = ema_fast - ema_slow
    df['MACD_Signal'] = df['MACD'].ewm(span=macd_signal, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']
    
    return df

In [22]:
# Пример использования
ticker = "LKOH"
df = get_5min(ticker)

# Добавляем индикаторы
df_with_indicators = add_technical_indicators(df)

# Теперь в df_with_indicators есть:
# SMA_10, SMA_20, SMA_50
# EMA_12, EMA_26
# RSI_14
# MACD, MACD_Signal, MACD_Hist

print(df_with_indicators.tail())

Загружаю из кэша: LKOH
                   DateTime   Open   High    Low  Close  Volume  SMA_10  \
4 2025-11-03 14:50:00+00:00  26.66  26.66  26.57  26.57    4003  26.682   
3 2025-11-03 14:45:00+00:00  26.66  26.68  26.60  26.68    8727  26.661   
2 2025-11-03 14:40:00+00:00  26.59  26.67  26.55  26.67    6642  26.651   
1 2025-11-03 14:35:00+00:00  26.53  26.62  26.38  26.61   26204  26.644   
0 2025-11-03 14:30:00+00:00  26.83  26.85  26.47  26.49   20881  26.631   

    SMA_20   SMA_50     EMA_12     EMA_26     RSI_14      MACD  MACD_Signal  \
4  26.8215  26.9830  26.708986  26.828842  27.069019 -0.119856    -0.100277   
3  26.8020  26.9734  26.704527  26.817817  27.696963 -0.113290    -0.102880   
2  26.7845  26.9624  26.699215  26.806868  28.537712 -0.107653    -0.103835   
1  26.7635  26.9514  26.685490  26.792285  29.093518 -0.106795    -0.104427   
0  26.7360  26.9384  26.655414  26.769893  28.878750 -0.114479    -0.106437   

   MACD_Hist  
4  -0.019579  
3  -0.010410  
2  -0.

In [18]:
df = get_5min(ticker)
df.head(20)

Скачиваю GOLD с 01.01.2025...
  +5 свечей → 2025-01-08 00:00
  +5 свечей → 2025-01-15 00:00
  +5 свечей → 2025-01-22 00:00
  +6 свечей → 2025-01-29 00:00
  +6 свечей → 2025-02-05 00:00
  +6 свечей → 2025-02-12 00:00
  +5 свечей → 2025-02-19 00:00
  +6 свечей → 2025-02-26 00:00
  +6 свечей → 2025-03-05 00:00
  +6 свечей → 2025-03-12 00:00
  +6 свечей → 2025-03-19 00:00
  +6 свечей → 2025-03-26 00:00
  +6 свечей → 2025-04-02 00:00
  +6 свечей → 2025-04-09 00:00
  +6 свечей → 2025-04-16 00:00
  +5 свечей → 2025-04-23 00:00
  +6 свечей → 2025-04-30 00:00
  +6 свечей → 2025-05-07 00:00
  +6 свечей → 2025-05-14 00:00
  +6 свечей → 2025-05-21 00:00
  +5 свечей → 2025-05-28 00:00
  +6 свечей → 2025-06-04 00:00
  +6 свечей → 2025-06-11 00:00
  +6 свечей → 2025-06-18 00:00
  +5 свечей → 2025-06-25 00:00
  +6 свечей → 2025-07-02 00:00
  +5 свечей → 2025-07-09 00:00
  +6 свечей → 2025-07-16 00:00
  +6 свечей → 2025-07-23 00:00
  +6 свечей → 2025-07-30 00:00
  +6 свечей → 2025-08-06 00:00
  +6 свеч

,DateTime,Open,High,Low,Close,Volume
286,2025-12-10 00:00:00+00:00,26.42,26.61,26.20,26.29,3614
285,2025-12-09 00:00:00+00:00,25.70,26.92,24.87,26.24,20022600
284,2025-12-09 00:00:00+00:00,25.70,26.92,24.87,26.24,20022600
283,2025-12-08 00:00:00+00:00,30.62,31.06,30.26,30.85,3101300
282,2025-12-06 00:00:00+00:00,30.16,30.16,30.16,30.16,0
281,2025-12-05 00:00:00+00:00,30.00,30.77,29.76,30.66,2310300
280,2025-12-04 00:00:00+00:00,29.97,30.39,29.97,30.16,1977900
279,2025-12-03 00:00:00+00:00,29.78,30.44,29.69,30.10,1846500
278,2025-12-02 00:00:00+00:00,29.88,30.13,29.64,29.78,2137000
277,2025-12-02 00:00:00+00:00,29.88,30.13,29.64,29.78,2137000


In [15]:
df.head(30)

,DateTime,Open,High,Low,Close,Volume
1641,2025-12-10 14:00:00+00:00,26.26,26.61,26.12,26.19,15439
1640,2025-12-10 00:00:00+00:00,26.25,26.25,26.25,26.25,0
1639,2025-12-09 20:00:00+00:00,26.36,30.85,26.14,26.24,32899994
1638,2025-12-09 19:00:00+00:00,26.57,26.92,26.00,26.34,5959607
1637,2025-12-09 18:00:00+00:00,26.63,26.67,26.36,26.57,13923963
1636,2025-12-09 17:00:00+00:00,26.03,26.74,26.03,26.63,9053684
1635,2025-12-09 16:00:00+00:00,25.82,26.06,25.75,26.03,10671881
1634,2025-12-09 15:00:00+00:00,25.58,25.96,25.16,25.82,11837076
1633,2025-12-09 14:00:00+00:00,30.85,30.85,24.87,25.58,8814008
1632,2025-12-09 00:00:00+00:00,29.82,29.82,29.82,29.82,0


Капитализация = Цена закрытия × Количество акций в свободном обращении (free-float)

In [9]:
import yfinance as yf

stock = yf.Ticker('SBER.ME')
stock.info
# tickers = ['SBER.ME', 'GAZP.ME', 'ROSN.ME', 'NVTK.ME', 'GMKN.ME',
#            'LKOH.ME', 'PLZL.ME', 'YNDX.ME', 'SNGS.ME', 'SNGSP.ME']  # Пример тикеров для MOEX

# data = {}
# for ticker in tickers:
#     stock = yf.Ticker(ticker)
#     info = stock.info
#     data[ticker] = info.get('marketCap', None)

# # Отсортировать по капитализации
# # sorted_tickers = sorted(data.items(), key=lambda x: x[1], reverse=True)
# for ticker, cap in data.items():
#     print(f"{ticker}: {cap}")

{'address1': '19 Vavilova Street',
 'city': 'Moscow',
 'zip': '117312',
 'country': 'Russia',
 'phone': '7 495 957 5731',
 'fax': '7 495 747 3731',
 'website': 'https://www.sberbank.ru',
 'industry': 'Banks - Regional',
 'industryKey': 'banks-regional',
 'industryDisp': 'Banks - Regional',
 'sector': 'Financial Services',
 'sectorKey': 'financial-services',
 'sectorDisp': 'Financial Services',
 'longBusinessSummary': 'Sberbank of Russia, together with its subsidiaries, engages in corporate and retail banking to corporate customers, large, medium, and small businesses, government bodies, and financial organizations in the Russian Federation. It operates through B2B (business to business); B2C (business to clients): Other segments.The company offers online banking; deposit and corporate structured products; ending products to corporate clients comprising lending, trade and export financing, interbank and overdraft lending, REPO, leasing, and other financing instruments; lending products 